
Input: `outputs/step3_models.joblib`.
Output: `outputs/validation_predictions.csv`, figure `slide_figures/pred.png`.

In [ ]:
# Working folder: holds HABs_ScrippsPier_v3.xlsx and the outputs/ folder shared by all steps.
# In Colab the folder is on Google Drive, so every step notebook can read what the previous step saved.
import os
WORK_FOLDER = "/content/drive/MyDrive/BIF633_Progress2"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK_FOLDER, exist_ok=True)
    os.chdir(WORK_FOLDER)
except ImportError:
    pass  # not running in Colab: use the current folder
os.makedirs("outputs", exist_ok=True)
print("Working folder:", os.getcwd())

In [ ]:
# Load everything saved by the previous step
import joblib
globals().update(joblib.load("outputs/step3_models.joblib"))
print("Loaded outputs/step3_models.joblib")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

# The models predict log(1 + Chl-a). Convert back to mg/m3 before scoring.
def to_mg(log_values):
    return np.clip(np.expm1(np.asarray(log_values)), 0, None)  # same as the team's convert_log_prediction_to_original

def rmse(observed, predicted):
    return np.sqrt(mean_squared_error(observed, predicted))

def mae(observed, predicted):
    return mean_absolute_error(observed, predicted)

# Observed Chl-a (mg/m3) for the training and validation weeks
observed_train = to_mg(y_train)
observed_validation = to_mg(y_validation)

# The three core models and the feature set used for "set C" by each model
core_models = ["Linear Regression", "Random Forest", "SVR"]
set_C_name = {
    "Linear Regression": "C_Linear_Selected",   # 13 features kept after the VIF check
    "Random Forest": "C_Full_Temporal",
    "SVR": "C_Full_Temporal",
}
short_name = {
    "A_Environmental": "A",
    "B_Environmental_Seasonal": "B",
    "C_Full_Temporal": "C",
    "C_Linear_Selected": "C",
}
print("Ready")

In [ ]:
rf_prediction = to_mg(trained_models["Random Forest__C_Full_Temporal"]["validation_prediction_log"])

plt.figure(figsize=(10, 4.5))
plt.plot(pd.to_datetime(validation_df["time"]), observed_validation, label="Observed", color="purple")
plt.plot(pd.to_datetime(validation_df["time"]), rf_prediction, label="Random Forest prediction", color="teal")
plt.ylabel("Chl-a (mg/m3)")
plt.legend()
plt.tight_layout()
plt.show()

print("Highest prediction:", round(rf_prediction.max(), 2), "mg/m3")
print("Highest observed:  ", round(observed_validation.max(), 2), "mg/m3")

In [ ]:
pd.DataFrame({
    "time": pd.to_datetime(validation_df["time"]).dt.strftime("%Y-%m-%d"),
    "Observed": observed_validation,
    "Random Forest": rf_prediction,
}).to_csv("outputs/validation_predictions.csv", index=False)
print("Saved outputs/validation_predictions.csv")

In [ ]:
import os
import matplotlib.dates as mdates
os.makedirs("slide_figures", exist_ok=True)  # figures go here
plt.rcParams.update({"font.size": 18, "axes.spines.top": False, "axes.spines.right": False})
PLUM, TEAL, OLIVE, GREY = "#8A3560", "#00958F", "#A4983A", "#B3B0AA"


# Slide 5: observed vs predicted (Random Forest, set C)
fig, ax = plt.subplots(figsize=(9.1, 4.9))
dates = pd.to_datetime(validation_df["time"])
ax.plot(dates, observed_validation, color=PLUM, linewidth=2.4, label="Observed")
ax.plot(dates, rf_prediction, color=TEAL, linewidth=2.4, label="Random Forest")
ax.set_ylabel("Chl-a (mg/m³)")
ax.set_ylim(0, 30)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.yaxis.grid(True, color="#E4E1DC")
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", ncol=2)
plt.tight_layout()
plt.savefig("slide_figures/pred.png", dpi=200)
plt.show()